# 02 — Data Cleaning

## Purpose

This notebook prepares the Smart TV product metadata and consumer reviews for analysis.

The cleaning process includes:
- Identifying Smart TVs from product metadata
- Restricting the analytical period to 2015–2023
- Extracting product attributes
- Cleaning review text and removing exact duplicates
- Validating the final product and review samples

The resulting datasets are used in the exploratory analysis, consumer review analysis, and competitive positioning notebooks.

In [8]:
import pandas as pd
import numpy as np
import re

products = pd.read_parquet("../data/tv_products.parquet")
reviews = pd.read_parquet("../data/tv_reviews.parquet")

print("Products:", products.shape)
print("Reviews:", reviews.shape)

Products: (3027, 17)
Reviews: (218226, 10)


## Initial Data Inspection

Inspect the product and review datasets to understand their structure, missing values, and rating distribution before cleaning.

In [9]:
# Product data overview
print("=== PRODUCT DATA ===")
products.info()

print("\nMissing values:")
display(
    products.isna()
    .sum()
    .sort_values(ascending=False)
)

=== PRODUCT DATA ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3027 entries, 0 to 3026
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   main_category    2843 non-null   object 
 1   title            3027 non-null   object 
 2   average_rating   3027 non-null   float64
 3   rating_number    3027 non-null   int64  
 4   features         3027 non-null   object 
 5   description      3027 non-null   object 
 6   price            3027 non-null   object 
 7   images           3027 non-null   object 
 8   videos           3027 non-null   object 
 9   store            3027 non-null   object 
 10  categories       3027 non-null   object 
 11  details          3027 non-null   object 
 12  parent_asin      3027 non-null   object 
 13  bought_together  0 non-null      object 
 14  subtitle         0 non-null      object 
 15  author           0 non-null      object 
 16  brand            3027 non-null   object

author             3027
subtitle           3027
bought_together    3027
main_category       184
store                 0
parent_asin           0
details               0
categories            0
videos                0
title                 0
images                0
price                 0
description           0
features              0
rating_number         0
average_rating        0
brand                 0
dtype: int64

In [10]:
# Review data overview
print("=== REVIEW DATA ===")
reviews.info()

print("\nMissing values:")
display(
    reviews.isna()
    .sum()
    .sort_values(ascending=False)
)

=== REVIEW DATA ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 218226 entries, 0 to 218225
Data columns (total 10 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   rating             218226 non-null  float64
 1   title              218226 non-null  object 
 2   text               218226 non-null  object 
 3   asin               218226 non-null  object 
 4   parent_asin        218226 non-null  object 
 5   user_id            218226 non-null  object 
 6   timestamp          218226 non-null  int64  
 7   helpful_vote       218226 non-null  int64  
 8   verified_purchase  218226 non-null  bool   
 9   brand              218226 non-null  object 
dtypes: bool(1), float64(1), int64(2), object(6)
memory usage: 15.2+ MB

Missing values:


rating               0
title                0
text                 0
asin                 0
parent_asin          0
user_id              0
timestamp            0
helpful_vote         0
verified_purchase    0
brand                0
dtype: int64

In [11]:
print("=== RATING DISTRIBUTION ===")

rating_distribution = (
    reviews["rating"]
    .value_counts()
    .sort_index()
)

display(rating_distribution)

print("\nPercent:")
display(
    (reviews["rating"]
     .value_counts(normalize=True)
     .sort_index() * 100)
    .round(2)
)

=== RATING DISTRIBUTION ===


rating
1.0     30812
2.0     11396
3.0     13848
4.0     31187
5.0    130983
Name: count, dtype: int64


Percent:


rating
1.0    14.12
2.0     5.22
3.0     6.35
4.0    14.29
5.0    60.02
Name: proportion, dtype: float64

## Product Metadata Cleaning

Remove metadata fields that are not needed for the analysis while retaining product identifiers, brand information, ratings, and attributes used in subsequent steps.

In [12]:
products_clean = products.drop(
    columns=[
        "author",
        "subtitle",
        "bought_together",
        "images",
        "videos"
    ]
).copy()

print(products_clean.shape)
print(products_clean.columns.tolist())

(3027, 12)
['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'store', 'categories', 'details', 'parent_asin', 'brand']


### Price Data Availability

Convert product prices to numeric values and assess missingness.

Because valid price information is available for only a small proportion of products, price is not used as a primary explanatory variable in the subsequent analysis. Consumer perceptions of value are instead examined through review text.

In [13]:
products_clean["price_numeric"] = pd.to_numeric(
    products_clean["price"],
    errors="coerce"
)

print("Valid prices:",
      products_clean["price_numeric"].notna().sum())

print("Missing/unusable prices:",
      products_clean["price_numeric"].isna().sum())

print("Coverage:",
      round(
          products_clean["price_numeric"].notna().mean() * 100,
          2
      ),
      "%"
)

products_clean["price_numeric"].describe()

Valid prices: 280
Missing/unusable prices: 2747
Coverage: 9.25 %


count      280.000000
mean      1580.365964
std       2128.141655
min         45.550000
25%        499.972500
50%        999.495000
75%       1865.367500
max      24996.990000
Name: price_numeric, dtype: float64

## Smart TV Identification

Identify Smart TVs using product titles, features, descriptions, and technical details rather than relying on product titles alone.

Smart TV identification is refined in subsequent steps to define the final analytical sample.

In [14]:
def combine_product_text(row):
    fields = [
        row["title"],
        row["features"],
        row["description"],
        row["details"]
    ]

    return " ".join(
        str(x) for x in fields
        if x is not None
    ).lower()


products_clean["product_text"] = products_clean.apply(
    combine_product_text,
    axis=1
)

In [15]:
smart_patterns = [
    r"\bsmart tv\b",
    r"\bsmart television\b",
    r"\broku tv\b",
    r"\bgoogle tv\b",
    r"\bandroid tv\b",
    r"\bfire tv\b",
    r"\bwebos\b",
    r"\btizen\b",
    r"\bsmartcast\b",
    r"\bnetflix\b",
    r"\bhulu\b",
    r"\bstreaming\b",
    r"\balexa\b",
    r"\bgoogle assistant\b",
    r"\bwi[\-\s]?fi\b"
]

smart_regex = "|".join(smart_patterns)

products_clean["is_smart_tv"] = (
    products_clean["product_text"]
    .str.contains(
        smart_regex,
        case=False,
        regex=True,
        na=False
    )
)

### Model Year Extraction

Extract model years from product metadata and titles, prioritizing the year identified in the title when available.

The extracted year is used to restrict the final analytical sample to 2015–2023.

In [16]:
def extract_year(text):
    matches = re.findall(
        r"\b(20(?:0[5-9]|1[0-9]|2[0-3]))\b",
        str(text)
    )

    if matches:
        return int(max(matches))

    return np.nan


products_clean["model_year"] = (
    products_clean["product_text"]
    .apply(extract_year)
)

print("Products with identifiable year:",
      products_clean["model_year"].notna().sum())

print(
    "Coverage:",
    round(
        products_clean["model_year"].notna().mean() * 100,
        2
    ),
    "%"
)

products_clean["model_year"].describe()

Products with identifiable year: 2815
Coverage: 93.0 %


count    2815.000000
mean     2015.307993
std         5.157091
min      2005.000000
25%      2011.000000
50%      2016.000000
75%      2020.000000
max      2023.000000
Name: model_year, dtype: float64

In [17]:
def extract_year_from_title(title):
    if not isinstance(title, str):
        return np.nan

    years = re.findall(
        r"\b(20(?:0[5-9]|1[0-9]|2[0-3]))\b",
        title
    )

    if years:
        return int(max(years))

    return np.nan


products_clean["title_year"] = (
    products_clean["title"]
    .apply(extract_year_from_title)
)

In [18]:
products_clean["year"] = (
    products_clean["title_year"]
    .fillna(products_clean["model_year"])
)

print("Year coverage:")
print(
    round(
        products_clean["year"].notna().mean() * 100,
        2
    ),
    "%"
)

products_clean["year"].describe()

Year coverage:
93.0 %


count    2815.000000
mean     2015.263943
std         5.126436
min      2005.000000
25%      2011.000000
50%      2016.000000
75%      2020.000000
max      2023.000000
Name: year, dtype: float64

In [19]:
strong_smart_patterns = [
    r"\bsmart tv\b",
    r"\bsmart television\b",
    r"\broku tv\b",
    r"\bgoogle tv\b",
    r"\bandroid tv\b",
    r"\bfire tv\b",
    r"\bwebos\b",
    r"\btizen\b",
    r"\bsmartcast\b"
]

strong_smart_regex = "|".join(strong_smart_patterns)

products_clean["strong_smart_signal"] = (
    products_clean["product_text"]
    .str.contains(
        strong_smart_regex,
        case=False,
        regex=True,
        na=False
    )
)

In [20]:
smart_products_final = products_clean[
    products_clean["year"].between(2015, 2023)
    & products_clean["strong_smart_signal"]
].copy()

print("Final products:", len(smart_products_final))

display(
    smart_products_final["brand"]
    .value_counts()
)

Final products: 1097


brand
Samsung    399
LG         308
Sony       204
TCL         94
Hisense     92
Name: count, dtype: int64

In [21]:
reviews_clean = reviews.copy()

reviews_clean["review_date"] = pd.to_datetime(
    reviews_clean["timestamp"],
    unit="ms"
)

reviews_clean["review_year"] = (
    reviews_clean["review_date"].dt.year
)

print(
    reviews_clean["review_date"].min(),
    "to",
    reviews_clean["review_date"].max()
)

display(
    reviews_clean["review_year"]
    .value_counts()
    .sort_index()
)

1999-12-31 03:58:26 to 2023-09-12 22:20:47.291000


review_year
1999        1
2000       13
2001        6
2002       15
2003       34
2004       98
2005      243
2006      912
2007     3132
2008     4492
2009     5493
2010     6120
2011     7678
2012     8268
2013    10969
2014    16688
2015    19252
2016    20095
2017    15041
2018    16438
2019    18593
2020    22784
2021    19577
2022    16738
2023     5546
Name: count, dtype: int64

In [22]:
final_products = smart_products_final.copy()

print("Final products:", len(final_products))

display(
    final_products["brand"]
    .value_counts()
)

Final products: 1097


brand
Samsung    399
LG         308
Sony       204
TCL         94
Hisense     92
Name: count, dtype: int64

In [23]:
final_asins = set(final_products["parent_asin"])

final_reviews = reviews_clean[
    reviews_clean["parent_asin"].isin(final_asins)
].copy()

# Align reviews with the analytical period
final_reviews = final_reviews[
    final_reviews["review_year"].between(2015, 2023)
].copy()

print("Final reviews:", len(final_reviews))

display(
    final_reviews["brand"]
    .value_counts()
)

Final reviews: 112526


brand
TCL        41976
Samsung    33084
LG         15315
Sony       12349
Hisense     9802
Name: count, dtype: int64

In [24]:
final_reviews["text_clean"] = (
    final_reviews["text"]
    .astype(str)
    .str.strip()
)

final_reviews["text_length"] = (
    final_reviews["text_clean"]
    .str.len()
)

print(
    "Empty before removal:",
    (final_reviews["text_length"] == 0).sum()
)

final_reviews = final_reviews[
    final_reviews["text_length"] > 0
].copy()

print(
    "Reviews after empty-text removal:",
    len(final_reviews)
)

Empty before removal: 233
Reviews after empty-text removal: 112293


In [25]:
before = len(final_reviews)

final_reviews = (
    final_reviews
    .drop_duplicates()
    .copy()
)

removed = before - len(final_reviews)

print("Exact duplicates removed:", removed)
print("Reviews remaining:", len(final_reviews))

Exact duplicates removed: 1102
Reviews remaining: 111191


In [26]:
potential_duplicates = final_reviews.duplicated(
    subset=[
        "user_id",
        "parent_asin",
        "text_clean"
    ],
    keep=False
)

print(
    "Potential duplicates remaining:",
    potential_duplicates.sum()
)

Potential duplicates remaining: 6


In [27]:
reviews_per_product = (
    final_reviews
    .groupby(
        ["brand", "parent_asin"]
    )
    .size()
    .reset_index(name="review_count")
)

display(
    reviews_per_product
    .groupby("brand")["review_count"]
    .agg(
        products="count",
        mean="mean",
        median="median",
        max="max"
    )
    .round(1)
)

,products,mean,median,max
brand,,,,
Hisense,92,104.9,17.5,635
LG,308,49.1,4.0,1217
Samsung,399,81.9,3.0,3661
Sony,203,60.1,3.0,906
TCL,94,442.1,22.5,6268


In [28]:
top_products = (
    reviews_per_product
    .sort_values(
        ["brand", "review_count"],
        ascending=[True, False]
    )
    .groupby("brand")
    .head(5)
)

top_products = top_products.merge(
    final_products[
        ["parent_asin", "title"]
    ],
    on="parent_asin",
    how="left"
)

display(
    top_products[
        [
            "brand",
            "title",
            "review_count"
        ]
    ]
)

,brand,title,review_count
0,Hisense,Hisense 50-Inch Class H8 Quantum Series Androi...,635
1,Hisense,Hisense 50A6G 50-Inch 4K Ultra HD Android Smar...,634
2,Hisense,Hisense 32-Inch Class H4 Series LED Roku Smart...,571
3,Hisense,Hisense 50-inch ULED U6HF Series Quantum Dot Q...,485
4,Hisense,Hisense 50-Inch Class R6 Series Dolby Vision H...,462
5,LG,"LG OLED C1 Series 55"" Alexa Built-in 4k Smart...",1217
6,LG,"LG OLED55CXPUA Alexa Built-In CX 55"" 4K Smart ...",820
7,LG,"LG 55UM7300PUA Alexa Built-in 55"" 4K Ultra HD ...",753
8,LG,"LG LED TV 22"" Full HD 1080p IPS Display, 60Hz ...",648
9,LG,LG Electronics 49UJ6300 49-Inch 4K Ultra HD Sm...,631


In [29]:
products_with_reviews = set(
    final_reviews["parent_asin"]
)

no_review_products = final_products[
    ~final_products["parent_asin"]
    .isin(products_with_reviews)
]

display(
    no_review_products[
        ["brand", "title", "year"]
    ]
)

,brand,title,year
494,Sony,Sony XR55A80K Bravia XR A80K 55 inch 4K HDR OL...,2022.0


In [30]:
final_products = final_products.drop(
    columns=[
        "product_text",
        "title_year",
        "model_year"
    ],
    errors="ignore"
)

In [31]:
print("=== FINAL PRODUCTS ===")
print(final_products.shape)
print(final_products.columns.tolist())

print("\n=== FINAL REVIEWS ===")
print(final_reviews.shape)
print(final_reviews.columns.tolist())

=== FINAL PRODUCTS ===
(1097, 16)
['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'store', 'categories', 'details', 'parent_asin', 'brand', 'price_numeric', 'is_smart_tv', 'year', 'strong_smart_signal']

=== FINAL REVIEWS ===
(111191, 14)
['rating', 'title', 'text', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase', 'brand', 'review_date', 'review_year', 'text_clean', 'text_length']


In [32]:
cleaning_summary = pd.DataFrame({
    "Metric": [
        "Raw target-brand TV products",
        "Final Smart TV products",
        "Raw TV reviews",
        "Final usable reviews",
        "Review period",
        "Price coverage",
        "Exact duplicate reviews removed"
    ],
    "Value": [
        "3,027",
        f"{len(final_products):,}",
        "218,226",
        f"{len(final_reviews):,}",
        "2015–2023",
        "9.25%",
        "1,102"
    ]
})

display(cleaning_summary)

,Metric,Value
0,Raw target-brand TV products,"3,027"
1,Final Smart TV products,"1,097"
2,Raw TV reviews,"218,226"
3,Final usable reviews,"111,191"
4,Review period,2015–2023
5,Price coverage,9.25%
6,Exact duplicate reviews removed,"1,102"


In [33]:
final_products.to_parquet(
    "../data/tv_products_clean.parquet",
    index=False
)

final_reviews.to_parquet(
    "../data/tv_reviews_clean.parquet",
    index=False
)

print("Saved:")
print("../data/tv_products_clean.parquet", final_products.shape)
print("../data/tv_reviews_clean.parquet", final_reviews.shape)

Saved:
../data/tv_products_clean.parquet (1097, 16)
../data/tv_reviews_clean.parquet (111191, 14)


In [34]:
products_check = pd.read_parquet(
    "../data/tv_products_clean.parquet"
)

reviews_check = pd.read_parquet(
    "../data/tv_reviews_clean.parquet"
)

print("Products:", products_check.shape)
print("Reviews:", reviews_check.shape)

Products: (1097, 16)
Reviews: (111191, 14)


## Final Analytical Sample

The raw dataset contained 3,027 television products and 218,226
associated reviews across Samsung, LG, Sony, TCL, and Hisense.

To align the dataset with the modern Smart TV market, I restricted the
analysis to products with a 2015–2023 year signal extracted from product
metadata and explicit evidence of Smart TV capability, such as Smart TV,
Roku TV, Google TV, Android TV, Fire TV, webOS, Tizen, or SmartCast.

Price was not used as a primary analytical variable because valid price
information was available for only 9.25% of products.

Bundle-related language was retained as a diagnostic flag rather than an
exclusion criterion because preliminary filtering removed a substantial
share of otherwise valid Smart TV listings.

Reviews were restricted to 2015–2023. Empty review texts and exact
duplicate rows were removed, while short but meaningful reviews such as
"Great" or "Love it" were retained.

The resulting analytical dataset contains 1,097 Smart TV products and
111,191 consumer reviews.